####  This Python Notebook file will contain the needed data preprocessing, ML Model training processes.

##  1. Problem Definition & Data Selection
**Problem Definition**: This assignment will focus on training a model that could accurately predict the sentiment expressed in a given sentence.  

**Data description**: Contains 31,000 text samples, labeled with the sentiments `neutral, negative, or positive`.   

**Dataset link**: https://www.kaggle.com/datasets/mabubakrsiddiq/sentiment-analysis-dataset-31k-samples


## 2. Data Preprocessing

### 2.1 Usual Preprocessing

In [1]:
import pandas as pd

df = pd.read_csv("sentiment_dataset.csv")
print('---HEAD----')
display(df.head())
print('---INFO + DATATYPES---')
display(df.info())

---HEAD----


,id,text,label,sentiment
0,0,"Cooking microwave pizzas, yummy",2,positive
1,1,Any plans of allowing sub tasks to show up in ...,1,neutral
2,2,"I love the humor, I just reworded it. Like sa...",2,positive
3,3,naw idk what ur talkin about,1,neutral
4,4,That sucks to hear. I hate days like that,0,negative


---INFO + DATATYPES---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31232 entries, 0 to 31231
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         31232 non-null  int64 
 1   text       31232 non-null  object
 2   label      31232 non-null  int64 
 3   sentiment  31232 non-null  object
dtypes: int64(2), object(2)
memory usage: 976.1+ KB


None

In [2]:
# Keep only what you need
# df = df[["id", "text", "label", "sentiment"]].copy()

# Basic cleaning
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0] # We don't want any line that would have no text (we have non-null values but just to always be sure)


print('---CHECK FOR DUPLICATES---')
print(f"Total duplicates: {df.duplicated(subset=['text']).sum()}")

---CHECK FOR DUPLICATES---
Total duplicates: 24


In [3]:

# Drop rows with missing target
df = df.dropna(subset=["sentiment", "label"]) # We don't want lines that are not labeled (we have non-null values but just to always be sure)

# drop exact duplicate texts for better performance
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True) # We found some duplicates values earlier after trimming

print(df.shape)
print(df["sentiment"].value_counts(dropna=False))

(31208, 4)
sentiment
neutral     11645
positive    10465
negative     9098
Name: count, dtype: int64


In [4]:
# Check if the `label` and `sentiment` columns always matched, no wrong dataset
# Then, we'll keep just one of these two features as label
ct = pd.crosstab(df["label"], df["sentiment"])
print(ct)

sentiment  negative  neutral  positive
label                                 
0              9098        0         0
1                 0    11645         0
2                 0        0     10465


In [5]:
df = df[["text", "sentiment"]].copy() # Keep only what features we need now

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

df["label_id"] = df["sentiment"].map(label2id).astype(int)
df_model = df[["text", "label_id"]].copy()


### 2.2 Tokenization for NLP Problems

In [6]:
from sklearn.model_selection import train_test_split
# Take the testing dataset 20%, then take the validation dataset 10% and the rest is for training 70%
train_df, test_df = train_test_split(
    df_model, test_size=0.2, random_state=42, stratify=df_model["label_id"]
)

train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df["label_id"]
)

print(train_df.shape, val_df.shape, test_df.shape)


(22469, 2) (2497, 2) (6242, 2)


In [7]:
train_df.head()

,text,label_id
16184,are you call that winning?,1
19608,Huh ? What`s wrong with UNIX environment dev`s ?,0
23490,Using this app is a great way to organize assi...,2
22470,"Syncing devices doesn't work, even after readi...",0
22658,Hi I have uploaded 5 completely new Chinese Le...,2


#### 1st Model (TF-IDF + Logistic Regression)
For the 1st model, we will use TF-IDF and Logistic Regression
- TF-IDF for turning text into numeric features
- Logistic Regression is a strong linear classifier that we'll use to classify the text features and predict the sentiment.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
import os

sk_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=200_000)),
    ("clf", LogisticRegression(max_iter=2000, n_jobs=None))
])

sk_model.fit(train_df["text"], train_df["label_id"])

val_pred = sk_model.predict(val_df["text"])
print("VALIDATION REPORT (sklearn):")
print(classification_report(val_df["label_id"], val_pred, digits=4))

test_pred = sk_model.predict(test_df["text"])
print("TEST REPORT (sklearn):")
print(classification_report(test_df["label_id"], test_pred, digits=4))

os.makedirs("artifacts", exist_ok=True)
joblib.dump(sk_model, "artifacts/sklearn_sentiment.joblib")
print("Saved -> artifacts/sklearn_sentiment.joblib")


VALIDATION REPORT (sklearn):
              precision    recall  f1-score   support

           0     0.6953    0.6236    0.6575       728
           1     0.6057    0.6642    0.6336       932
           2     0.7482    0.7348    0.7414       837

    accuracy                         0.6760      2497
   macro avg     0.6830    0.6742    0.6775      2497
weighted avg     0.6796    0.6760    0.6767      2497

TEST REPORT (sklearn):
              precision    recall  f1-score   support

           0     0.6808    0.6071    0.6419      1820
           1     0.5953    0.6582    0.6252      2329
           2     0.7490    0.7315    0.7401      2093

    accuracy                         0.6679      6242
   macro avg     0.6751    0.6656    0.6691      6242
weighted avg     0.6718    0.6679    0.6686      6242

Saved -> artifacts/sklearn_sentiment.joblib


#### 2nd Model : Fine-tuned DistilBERT for sentiment analysis
For the 2nd model, we picked a trained model `distilbert-base-uncased` which already learned General Language structure.
We'll fine-tune this model, using our dataset.

In [13]:
!pip install evaluate --quiet

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import os, json

# ============================================================
# DistilBERT fine-tuning (PyTorch backend via Hugging Face)
#
# What we do here:
# 1) Loads a pretrained DistilBERT tokenizer + model
# 2) Converts your pandas splits (train_df/val_df/test_df) into HF Datasets
# 3) Tokenizes text -> input_ids + attention_mask (+ labels)
# 4) Uses Trainer to run the PyTorch training loop automatically
# 5) Evaluates on test set
# 6) Saves model + tokenizer + label map for deployment
#
# Key idea:
# - DistilBERT IS a deep neural network.
# - You don't manually write forward/backward; Trainer handles the loop.
# ============================================================

# 1) Loads a pretrained DistilBERT tokenizer + model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def to_hf_dataset(df_): # 2) Convert our pandas dataset into Hugging Face Dataset
    ds = Dataset.from_pandas(df_[["text", "label_id"]].rename(columns={"label_id": "labels"}))
    return ds

train_ds = to_hf_dataset(train_df) # Convert them one by one
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

def tokenize_fn(batch): # Function for the tokenization using the `tokenizer of [distilbert-base-uncased]`
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

# 3) Tokenizes text -> input_ids + attention_mask (+ labels)
train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [24]:
# View train_ds first values
# train_ds[:3]
# train_df.head()

In [12]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_weighted": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

num_labels = 3
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# 4) Uses Trainer to run the PyTorch training loop automatically
args = TrainingArguments(
    output_dir="artifacts/tmp_trainer",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train() # Launch the trainer

print("FINAL TEST METRICS (transformer):")
print(trainer.evaluate(test_ds))

# Save artifacts for deployment
out_dir = "artifacts/transformer"
os.makedirs(out_dir, exist_ok=True)
trainer.model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)

with open("artifacts/label_map.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

print("Saved -> artifacts/transformer/ and artifacts/label_map.json")


Map:   0%|          | 0/22469 [00:00<?, ? examples/s]

Map:   0%|          | 0/2497 [00:00<?, ? examples/s]

Map:   0%|          | 0/6242 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.596642,0.589367,0.748098,0.747773
2,0.491897,0.600750,0.757709,0.758002


... (3 outputs left out)

FINAL TEST METRICS (transformer):


{'eval_loss': 0.6136849522590637, 'eval_accuracy': 0.7553668695930792, 'eval_f1_weighted': 0.7553755073806911, 'eval_runtime': 24.4968, 'eval_samples_per_second': 254.808, 'eval_steps_per_second': 8.001, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved -> artifacts/transformer/ and artifacts/label_map.json


### Real Test with user input

In [17]:
# Load saved sklearn pipeline
sk_model = joblib.load("artifacts/sklearn_sentiment.joblib")

def predict_sklearn(texts):
    """
    Returns:
      - pred_ids: predicted class ids
      - probs: probability per class (if supported)
    """
    pred_ids = sk_model.predict(texts)

    # LogisticRegression supports predict_proba
    if hasattr(sk_model, "predict_proba"):
        probs = sk_model.predict_proba(texts)
        return pred_ids, probs
    return pred_ids, None

# Try a few examples
examples = [
    "I love this product, it's amazing.",
    "I hate life so much, .....",
    "Grrrr, I don't like that."
]

pred_ids, probs = predict_sklearn(examples)

print("SKLEARN PRED IDS:", pred_ids)
if probs is not None:
    print("SKLEARN PROBS:\n", np.round(probs, 4))


SKLEARN PRED IDS: [2 0 0]
SKLEARN PROBS:
 [[0.0112 0.0204 0.9685]
 [0.899  0.0321 0.0689]
 [0.6414 0.219  0.1396]]


In [25]:
# Load label map
with open("artifacts/label_map.json", "r", encoding="utf-8") as f:
    maps = json.load(f)

# Sometimes JSON keys become strings; convert id2label keys to int safely
id2label_raw = maps["id2label"]
id2label = {int(k): v for k, v in id2label_raw.items()} if isinstance(next(iter(id2label_raw.keys())), str) else id2label_raw

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained("artifacts/transformer")
model = AutoModelForSequenceClassification.from_pretrained("artifacts/transformer")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_transformer(texts, max_length=128):
    """
    Returns:
      - pred_ids: predicted class ids
      - probs: list of probability vectors
      - labels: predicted label strings
    """
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)

    pred_ids = torch.argmax(probs, dim=-1).cpu().tolist()
    probs = probs.cpu().tolist()
    labels = [id2label[i] for i in pred_ids]
    return pred_ids, probs, labels

examples = [
    "I love this product, it's amazing.",
    "I hate life so much, .....",
    "Grrrr, I don't like that."
]

pred_ids, probs, labels = predict_transformer(examples, max_length=128)

print("TRANSFORMER PRED IDS:", pred_ids)
print("TRANSFORMER LABELS:", labels)
print("TRANSFORMER PROBS:")
for t, p in zip(examples, probs):
    print("-", t)
    print("  ", [round(x, 4) for x in p])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

TRANSFORMER PRED IDS: [2, 0, 0]
TRANSFORMER LABELS: ['positive', 'negative', 'negative']
TRANSFORMER PROBS:
- I love this product, it's amazing.
   [0.0033, 0.0098, 0.9869]
- I hate life so much, .....
   [0.9729, 0.0224, 0.0047]
- Grrrr, I don't like that.
   [0.9621, 0.0331, 0.0049]
